[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_13/MFM_ch13_purged_cv/MFM_ch13_purged_cv.ipynb)

# MFM_ch13_purged_cv

**Modelling Financial Markets (MFM) — Chapter 13: Machine Learning in Finance**

Purged K-Fold cross-validation with embargo: schematic of train/test/purge/embargo sets and a leakage experiment on a pure random walk with overlapping labels, where shuffled K-Fold reports spurious accuracy while purged K-Fold returns to a coin flip.

*Author: Daniel Traian Pele*

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip -q install shap

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## Style and data

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts

SEED = 42


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


In [ ]:
import os
DATA_URL = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
SERIES = {'sp500': 'GSPC.INDX', 'vix': 'VIX.INDX', 'btc': 'BTC-USD.CC'}
START = {'sp500': '2000-01-01', 'vix': '2000-01-01', 'btc': '2014-09-17'}


def load_data(name='sp500', start=None, end='2026-09-18'):
    """OHLCV zilnic din data/market: copia locala, altfel fisierul din repo-ul GitHub MFM."""
    fname = SERIES[name] + '.csv'
    local = [os.path.join(d, 'data', 'market', fname) for d in ('.', '..', '../..', '../../..')]
    path = next((p for p in local if os.path.exists(p)), DATA_URL + fname)
    df = pd.read_csv(path, parse_dates=['date']).set_index('date')
    df = df.rename(columns=str.capitalize)[['Open', 'High', 'Low', 'Close', 'Volume']]
    return df.loc[start or START[name]:end]


spx = load_data('sp500')['Close']
vix = load_data('vix')['Close']
btc = load_data('btc')['Close']
log_spx = np.log(spx)
print('S&P 500:', spx.index[0].date(), '->', spx.index[-1].date(), len(spx), 'obs')
print('BTC    :', btc.index[0].date(), '->', btc.index[-1].date(), len(btc), 'obs')

## Helper functions

In [ ]:
class PurgedKFold:
    """
    K-Fold fara amestecare pentru etichete care se suprapun in timp.

    t1 : pd.Series indexata dupa momentul observatiei t0, cu valoarea = momentul
         in care se cunoaste eticheta (sfarsitul ferestrei etichetei).
    Purjare: se elimina din antrenare observatiile al caror interval [t0, t1]
             se suprapune cu intervalul setului de test.
    Embargo: se elimina si o fractiune pct_embargo de observatii imediat dupa test.
    """

    def __init__(self, n_splits=5, t1=None, pct_embargo=0.01):
        self.n_splits = n_splits
        self.t1 = t1
        self.pct_embargo = pct_embargo

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        idx = np.arange(len(X))
        t0 = self.t1.index.values
        t1 = self.t1.values
        embargo = int(len(X) * self.pct_embargo)
        for test in np.array_split(idx, self.n_splits):
            start, stop = test[0], test[-1]
            test_t0, test_t1 = t0[start], t1[test].max()
            # purjare: antrenare doar pe observatii care nu se suprapun cu testul
            before = idx[(t1 < test_t0)]
            after_start = min(stop + 1 + embargo, len(X))
            after = idx[after_start:][t0[after_start:] > test_t1]
            yield np.concatenate([before, after]), test

## Charts

In [ ]:
def fig_purged_cv_scheme():
    n, k, h, emb = 100, 5, 4, 3
    fig, ax = plt.subplots(figsize=(7.0, 2.6))
    for fold in range(k):
        y = k - fold
        t_start, t_end = fold * n // k, (fold + 1) * n // k
        for t in range(n):
            if t_start <= t < t_end:
                c = IDAred
            elif t_start - h <= t < t_start:
                c = Purple
            elif t_end <= t < t_end + emb:
                c = Amber
            else:
                c = MainBlue
            ax.add_patch(mpatches.Rectangle((t, y - 0.35), 1, 0.7, fc=c, ec='none'))
        ax.text(-2, y, f'Fold {fold + 1}', ha='right', va='center', fontsize=8)
    ax.set_xlim(-12, n + 1)
    ax.set_ylim(0.3, k + 0.7)
    ax.set_yticks([])
    ax.set_xlabel('Time (observation index)')
    ax.spines['left'].set_visible(False)
    handles = [mpatches.Patch(color=MainBlue, label='Train'),
               mpatches.Patch(color=IDAred, label='Test'),
               mpatches.Patch(color=Purple, label='Purged (label overlaps test)'),
               mpatches.Patch(color=Amber, label='Embargo (after test)')]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=4, frameon=False)
    plt.tight_layout()
    save_fig('ch13_purged_cv_scheme')

In [ ]:
def fig_leakage_experiment():
    rng = np.random.default_rng(SEED)
    n, h = 3000, 20
    # pret: mers aleator pur -> nu exista nimic de prezis
    p = pd.Series(np.cumsum(rng.normal(0, 0.01, n + h)))
    y = (p.shift(-h) - p > 0).astype(int).iloc[:n].values          # eticheta pe h zile (suprapusa)
    # caracteristici persistente, dar independente de viitor
    X = np.column_stack([pd.Series(rng.normal(size=n + h)).rolling(h).mean().bfill().iloc[:n]
                         for _ in range(5)])
    t1 = pd.Series(np.arange(n) + h, index=np.arange(n))
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED)

    schemes = {
        'K-Fold\n(shuffled)': KFold(5, shuffle=True, random_state=SEED),
        'K-Fold\n(no shuffle)': KFold(5, shuffle=False),
        'Purged K-Fold\n+ embargo': PurgedKFold(5, t1=t1, pct_embargo=0.01),
    }
    res = {}
    for name, cv in schemes.items():
        accs = []
        for tr, te in cv.split(X):
            model.fit(X[tr], y[tr])
            accs.append(accuracy_score(y[te], model.predict(X[te])))
        res[name] = accs

    fig, ax = plt.subplots(figsize=(6.2, 2.9))
    colors = [IDAred, Amber, Forest]
    for i, (name, accs) in enumerate(res.items()):
        ax.bar(i, np.mean(accs), 0.55, color=colors[i], alpha=0.85)
        ax.scatter(np.full(len(accs), i) + np.linspace(-0.12, 0.12, len(accs)), accs,
                   color='black', s=10, zorder=3)
        ax.text(i + 0.3, np.mean(accs) + 0.01, f'{np.mean(accs):.1%}', ha='left', fontsize=8.5,
                fontweight='bold', color=colors[i])
    ax.axhline(0.5, color=Gray, ls='--', lw=0.8)
    ax.text(2.45, 0.505, 'coin flip', fontsize=7.5, color='black', ha='right', va='bottom')
    ax.set_xticks(range(len(res)), list(res.keys()))
    ax.set_ylim(0.35, 1.0)
    ax.set_ylabel('Out-of-fold accuracy')
    ax.set_title('Random walk + persistent noise features: there is NOTHING to predict',
                 fontsize=9, loc='left')
    plt.tight_layout()
    save_fig('ch13_leakage_experiment')
    return {k: np.mean(v) for k, v in res.items()}

## Run

In [ ]:
fig_purged_cv_scheme()
print(fig_leakage_experiment())

![ch13_purged_cv_scheme](./ch13_purged_cv_scheme.png)

![ch13_leakage_experiment](./ch13_leakage_experiment.png)

Output: `ch13_purged_cv_scheme.pdf`, `ch13_leakage_experiment.pdf`